# Erosion Prediction Pipeline

Incremental build of a modular pipeline:

1. **Load input features** — consistently load scope regions, vvr_rates_of_change, bank points, centerlines
2. **Visualize scope region** — river banks, line approximations, dist points
3. **Predict erosion speed** — from input features
4. **Predict coordinates** — year + erosion speed → bank position
5. **Distance to signaleringslijn** — per scope region
6. **Time to crossing** — years until bank reaches signaleringslijn
7. **Export layer** — predicted points + predicted_vvr_crossing_year

---
## Step 1: Load input features

Primary input: `wocu_post_processed_fase2_20260223.gpkg`
Raw input (for bank points, centerlines): `wocu_output_fase2_20260210.gpkg`

The processed file has scope and vvr but no lines; we derive dist points from raw punten_oever + centerlines.

### 1.1 Setup & paths

In [1]:
print('hello')

hello


In [ ]:
# Track lines of code (run after changes to see reduction)
try:
    from pathlib import Path
    from src.erosion.centerline_utils import count_notebook_loc
    _nb = Path.cwd() / "notebooks/04_model/20260313_erosion_prediction_pipeline.ipynb"
    if not _nb.exists():
        _nb = Path.cwd() / "20260313_erosion_prediction_pipeline.ipynb"
    _loc = count_notebook_loc(_nb)
    print(f"Notebook LOC: total={_loc['total']}, non_empty={_loc['non_empty']}  (baseline: 593 total, 528 non_empty)")
except Exception as e:
    print(f"LOC count skipped: {e}")

In [ ]:
# Ensure kernel cwd is backend/ so paths resolve correctly (run this first)
import os
import sys
from pathlib import Path
_cwd = Path.cwd()
if (_cwd / "src").exists():
    _backend = _cwd
elif (_cwd.parent / "src").exists():
    _backend = _cwd.parent
else:
    _backend = _cwd
os.chdir(_backend)
sys.path.insert(0, str(_backend))
sys.path.insert(0, str(Path.cwd()))

print("cwd:", os.getcwd())

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as mcm

import src.paths as PATHS
import src.model.baseline_model as BM
from src.erosion.centerline_utils import (
    build_ref_geom_lookup,
    compute_qualifying_regions,
    compute_vvr_crossing_year,
    ensure_axes_list,
    ensure_location_id_column,
    flatten_geom_to_lines,
    get_nvo_location_ids,
    offset_line_toward,
    parallel_line_from_vvr,
    pick_across_clusters,
    point_from_offset,
)
from src.model.export_utils import load_model_bundle
from src.model.predictor import predict_iterative_baseline, predict_iterative_ml

DATA_DIR = PATHS.DATA_DIR
POSTPROC_GPKG = DATA_DIR / '02_processed/erosion/wocu_post_processed_fase2_20260223.gpkg'
RAW_GPKG     = DATA_DIR / '01_raw/erosion/wocu_output_fase2_20260210.gpkg'

N_POINTS_FOR_DIST = 3  # mean of N furthest OK points (like DataHandler / explore_point_selection)

### 1.2 Define `load_input_features()`

In [ ]:
def load_input_features(
    postproc_path: Path = POSTPROC_GPKG,
    raw_path: Path = RAW_GPKG,
) -> dict:
    """
    Load scope regions, vvr_rates_of_change, bank points, centerlines.
    Returns dict with: scope, vvr_rates_of_change, bank_points, centerlines.
    """
    scope = ensure_location_id_column(gpd.read_file(postproc_path, layer='summary_scope'))
    vvr = gpd.read_file(postproc_path, layer='vvr_rates_of_change')
    bank_points = ensure_location_id_column(gpd.read_file(raw_path, layer='punten_oever'))
    centerlines = ensure_location_id_column(gpd.read_file(raw_path, layer='centrelines'))
    
    return {
        'scope': scope,
        'vvr_rates_of_change': vvr,
        'bank_points': bank_points,
        'centerlines': centerlines,
    }


### 1.3 Define `compute_dist_per_year()` — derive dist + point per (location_id, year)

In [ ]:
def compute_dist_per_year(
    bank_points: gpd.GeoDataFrame,
    centerlines: gpd.GeoDataFrame,
    n_points: int = N_POINTS_FOR_DIST,
    status_filter: str = 'OK',
) -> pd.DataFrame:
    """
    For each (location_id, year): dist from centerline, and point geometry.
    Uses N furthest OK points, mean of their dist. Point = midpoint of offset line.
    """
    cl_lookup = centerlines.set_index('location_id')['geometry'].to_dict()
    records = []
    for (loc_id, year), grp in bank_points.groupby(['location_id', 'dtm_date']):
        ok = grp[grp['status'] == status_filter]
        if ok.empty:
            continue
        chosen = ok.nlargest(n_points, 'dist')
        mean_dist = chosen['dist'].mean()
        cline = cl_lookup.get(loc_id)
        point = point_from_offset(cline, mean_dist, chosen.geometry) if cline is not None else None
        records.append({
            'location_id': loc_id,
            'year': year,
            'dist_m': mean_dist,
            'point': point,
        })
    df = pd.DataFrame(records)
    return df

### 1.4 Load and confirm

In [ ]:
data = load_input_features()
scope = data['scope']
vvr = data['vvr_rates_of_change']
bank_points = data['bank_points']
centerlines = data['centerlines']

print('Scope regions:', len(scope))
print('vvr_rates_of_change:', len(vvr))
print('Bank points:', len(bank_points), '| locations:', bank_points['location_id'].nunique())
print('Centerlines:', len(centerlines))

In [ ]:
dist_per_year = compute_dist_per_year(bank_points, centerlines)
print('dist_per_year:', len(dist_per_year), 'rows')
print('Years per location (sample):', dist_per_year.groupby('location_id')['year'].apply(list).head(3).to_dict())
print('Sample (first 5):')
display(dist_per_year.head())

### 1.5 Sanity check: one region with 3 river banks + 3 dist points

In [ ]:
# Pick a location with 3 years
years_per_loc = dist_per_year.groupby('location_id')['year'].apply(lambda s: sorted(s.tolist()))
three_year_locs = years_per_loc[years_per_loc.apply(len) == 3].index.tolist()[:3]
print('Sample locations with 3 years:', three_year_locs)

loc_id = three_year_locs[0]
sub = dist_per_year[dist_per_year['location_id'] == loc_id].sort_values('year')
print(f'\n{loc_id}: years={sub["year"].tolist()}, dists={sub["dist_m"].round(2).tolist()}')
print('Points present:', sub['point'].notna().all())

In [ ]:
# dist_per_year with point coordinates for sample location
sub_coords = sub.copy()
sub_coords['x'] = sub_coords['point'].apply(lambda p: p.x if p is not None else np.nan)
sub_coords['y'] = sub_coords['point'].apply(lambda p: p.y if p is not None else np.nan)
display(sub_coords[['location_id', 'year', 'dist_m', 'x', 'y']])

---
## Step 2: Visualize dist (same as explore_point_selection)

Same scope regions and style as `20260311_explore_point_selection.ipynb`: bank points per year, N furthest OK highlighted, dashed offset line at mean dist.

In [ ]:
MIN_SHIFT = 5.0
N_POINTS = 3
CLUSTERS = ['rijn', 'ijssel', 'maas', 'neder']
YEAR_COLORS = ['#f4d03f', '#a50026', '#4393c3']
STATUS_ALPHA = {'OK': 0.80, 'UNCERTAIN': 0.40, 'OUTLIER': 0.20}

PRED_YEARS = list(range(2026, 2036))
pred_cmap = mcm.get_cmap('viridis')
pred_colors = [pred_cmap((y - 2026) / 9) for y in PRED_YEARS]

year_patches = [mpatches.Patch(color=YEAR_COLORS[i], label=f't{i+1} (hist)') for i in range(3)]
pred_patches = [mpatches.Patch(color=pred_colors[i], label=f'{PRED_YEARS[i]} (pred)') for i in [0, 4, 9]]

scope_raw = gpd.read_file(RAW_GPKG, layer='vlakken_scope')
scope_raw = ensure_location_id_column(scope_raw)
cl_lookup = centerlines.set_index('location_id')['geometry'].to_dict()
scope_lookup = scope_raw.set_index('location_id')['geometry'].to_dict()

qualifying = compute_qualifying_regions(bank_points, min_shift=MIN_SHIFT, n_points=N_POINTS)
selected = pick_across_clusters(qualifying.index.tolist(), clusters=CLUSTERS, n=4)
print(f'Selected regions (same as explore_point_selection): {selected}')



In [ ]:
def draw_region(ax, loc_id, col_idx=0, show_t1_to_t2_arrow=True, show_predictions=False,
                show_signalering=False, show_bank_points=True, show_parallel_line=False, compact=False):
    cline, sgeom = cl_lookup.get(loc_id), scope_lookup.get(loc_id)
    _setup_ax(ax, loc_id, col_idx, compact)
    _draw_scope_and_cl(ax, sgeom, cline, compact)
    offset_lines = _draw_bank_points(ax, loc_id, cline) if show_bank_points else []
    if show_t1_to_t2_arrow:
        _draw_arrows(ax, offset_lines)
    if show_predictions:
        _draw_predictions(ax, loc_id)
    if show_signalering or show_parallel_line:
        _draw_signalering(ax, sgeom, cline, show_parallel_line, compact)

def _draw_predictions(ax, loc_id):
    pred = predicted_bank_positions[
        (predicted_bank_positions['location_id'] == loc_id) &
        predicted_bank_positions.geometry.notna()
    ].sort_values('year')
    for _, row in pred.iterrows():
        if row['year'] in PRED_YEARS:
            idx = PRED_YEARS.index(row['year'])
            ax.scatter(row.geometry.x, row.geometry.y, c=[pred_colors[idx]], s=80,
                       edgecolors='black', lw=0.6, zorder=9, marker='s')

def _setup_ax(ax, loc_id, col_idx, compact):
    fs, fst = (7, 5) if compact else (8.5, 6)
    ax.set_aspect('equal')
    ax.set_title(loc_id, fontsize=fs, pad=3 if compact else 5)
    ax.tick_params(labelsize=fst)
    if col_idx == 0:
        ax.set_ylabel('Northing (m RD)', fontsize=8)
    ax.set_xlabel('Easting (m RD)', fontsize=8)

def _draw_scope_and_cl(ax, sgeom, cline, compact):
    lw_cl = 1.5 if compact else 2.5
    if sgeom is not None:
        bx, by = sgeom.exterior.xy
        ax.fill(bx, by, fc='#f0f0f0', ec='#aaaaaa', lw=0.8 if compact else 1.2, zorder=1)
    if cline is not None:
        xs, ys = cline.xy
        ax.plot(xs, ys, color='black', lw=lw_cl, zorder=5, solid_capstyle='round')
        ax.text(xs[-1], ys[-1], ' CL', fontsize=7, color='black', va='center', zorder=6)

def _draw_bank_points(ax, loc_id, cline):
    """Returns list of offset lines (one per date, None if missing)."""
    grp = bank_points[bank_points['location_id'] == loc_id].copy()
    offset_lines = []
    for d_idx, date in enumerate(sorted(grp['dtm_date'].unique())):
        color = YEAR_COLORS[d_idx]
        yr_grp = grp[grp['dtm_date'] == date]
        for status, sub in yr_grp.groupby('status'):
            ax.scatter(sub.geometry.x, sub.geometry.y, c=color, s=8,
                       alpha=STATUS_ALPHA.get(status, 0.3), linewidths=0, zorder=3)
        ok_grp = yr_grp[yr_grp['status'] == 'OK']
        chosen = ok_grp.nlargest(N_POINTS, 'dist')
        if chosen.empty or cline is None:
            offset_lines.append(None)
            if cline is not None:
                mid = cline.interpolate(0.5, normalized=True)
                ax.text(mid.x, mid.y, f'{date}\nno OK pts', fontsize=5.5, color=color,
                        ha='center', va='bottom', zorder=9,
                        bbox=dict(fc='white', ec=color, alpha=0.7, pad=1, lw=0.8))
            continue
        mean_dist = chosen['dist'].mean()
        ax.scatter(chosen.geometry.x, chosen.geometry.y, c=color, s=110,
                   edgecolors='black', lw=0.8, zorder=8)
        offset_line = offset_line_toward(cline, mean_dist, chosen.geometry)
        if offset_line is not None and not offset_line.is_empty:
            ox, oy = offset_line.xy
            ax.plot(ox, oy, color=color, lw=2.2, ls='--', zorder=6, alpha=0.95)
            mid = offset_line.interpolate(0.5, normalized=True)
            ax.text(mid.x, mid.y, f'{date}\n{mean_dist:.1f} m  ({len(ok_grp)}/{len(yr_grp)} OK)',
                    fontsize=5.5, color=color, ha='center', va='bottom', zorder=9,
                    bbox=dict(fc='white', ec='none', alpha=0.65, pad=1))
        offset_lines.append(offset_line if offset_line and not offset_line.is_empty else None)
    return offset_lines

def _draw_arrows(ax, offset_lines, arrow_offset=10.0):
    pairs = [(0, 1, 'red', arrow_offset, 1), (1, 2, '#4393c3', arrow_offset * 2, -1)]
    for i, j, color, offset, side in pairs:
        if len(offset_lines) <= j or offset_lines[i] is None or offset_lines[j] is None:
            continue
        p1 = offset_lines[i].interpolate(0.5, normalized=True)
        p2 = offset_lines[j].interpolate(0.5, normalized=True)
        dx, dy = p2.x - p1.x, p2.y - p1.y
        dn = np.hypot(dx, dy)
        ux, uy = (-dy / dn * side, dx / dn * side) if dn > 1e-6 else (0, 0)
        ax.annotate('', xy=(p2.x + offset * ux, p2.y + offset * uy),
                    xytext=(p1.x + offset * ux, p1.y + offset * uy),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2,
                                   linestyle=':', mutation_scale=25))

def _draw_signalering(ax, sgeom, cline, show_parallel_line, compact):
    if sgeom is None:
        return
    local_sig = signaleringslijn[signaleringslijn.intersects(sgeom.buffer(10) if show_parallel_line else sgeom)]
    for _, row in local_sig.iterrows():
        for part in flatten_geom_to_lines(row.geometry):
            try:
                ax.plot(*part.xy, color=VVR_COLOR, lw=1.5 if compact else 2.5, zorder=7, solid_capstyle='round')
            except (NotImplementedError, AttributeError):
                pass
    if show_parallel_line and not local_sig.empty and cline is not None:
        vvr_clipped = local_sig.geometry.union_all().intersection(sgeom)
        if vvr_clipped and not vvr_clipped.is_empty:
            pline = parallel_line_from_vvr(cline, vvr_clipped)
            if pline and not pline.is_empty:
                for part in flatten_geom_to_lines(pline):
                    try:
                        ax.plot(*part.xy, color=PARALLEL_LINE_COLOR, lw=1.2, ls='--', zorder=6)
                    except (NotImplementedError, AttributeError):
                        pass


In [ ]:
def plot_regions(
    loc_ids,
    title,
    legend_handles,
    draw_kwargs=None,
    per_region_xlabel=None,   # callable(loc_id) -> str, optional
    n_cols=None,
    fig_w=5.5,
    fig_h=8,
    compact=False,
):
    draw_kwargs = draw_kwargs or {}
    n = len(loc_ids)
    
    if compact:
        n_cols = n_cols or 10
        n_rows = math.ceil(n / n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.0 * n_cols, 2.0 * n_rows), constrained_layout=True)
        axes_flat = np.array(axes).flatten()
    else:
        fig, axes = plt.subplots(1, n, figsize=(fig_w * n, fig_h), constrained_layout=True)
        axes_flat = ensure_axes_list(axes)

    for col, loc_id in enumerate(loc_ids):
        draw_region(axes_flat[col], loc_id, col_idx=col, compact=compact, **draw_kwargs)
        if per_region_xlabel:
            axes_flat[col].set_xlabel(per_region_xlabel(loc_id), fontsize=7.5)

    fig.legend(handles=legend_handles, loc='lower center', ncol=min(len(legend_handles), 9),
               fontsize=8 if not compact else 9, bbox_to_anchor=(0.5, -0.02 if compact else -0.06), frameon=True)
    fig.suptitle(title, fontsize=11, y=1.01 if compact else 1.02)
    plt.show()



In [ ]:
legend_extra = [
    plt.Line2D([0], [0], color='black', lw=2.5, label='centreline'),
    mpatches.Patch(fc='#f0f0f0', ec='#aaaaaa', label='scope boundary'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='grey',
               markeredgecolor='black', markersize=8, label=f'{N_POINTS} furthest OK (selected)'),
    mpatches.Patch(color='grey', alpha=0.80, label='opacity: OK'),
    mpatches.Patch(color='grey', alpha=0.40, label='opacity: UNCERTAIN'),
    mpatches.Patch(color='grey', alpha=0.20, label='opacity: OUTLIER'),
]

plot_regions(selected, title='Bank position per year...', legend_handles=year_patches + legend_extra,
    draw_kwargs=dict(show_t1_to_t2_arrow=True),
    per_region_xlabel=lambda lid: (
        f'Easting (m RD)\n'
        f'delta t1 to t2 = {qualifying.loc[lid]["shift_t1t2"]:.1f} m   '
        f'delta t2 to t3 = {qualifying.loc[lid]["shift_t2t3"]:.1f} m'
    )
)

---
## Step 3: Predict erosion speed

Load the baseline model and build `start_points` from `dist_per_year`, then run `predict_iterative_baseline`.
The pipeline uses the baseline here; swap to `predict_iterative_ml` to use an ML bundle instead.

In [ ]:
START_YEAR       = 2026
END_YEAR         = 2035
STEP             = 1
PREDICTION_YEARS = list(range(START_YEAR, END_YEAR + 1, STEP))

# --- Load baseline model ---
MODEL_PKL = DATA_DIR / '04_model_outputs/20260217/baseline_model_full_dataset.pkl'
baseline = BM.BaselineErosionModel.load_model(MODEL_PKL)
model_velocities = baseline.model
print(f'Baseline model: {len(model_velocities):,} locations')

# --- Build start_points from dist_per_year ---
# Last observed (dist, year) per location
last_obs = (
    dist_per_year.sort_values('year')
    .groupby('location_id')
    .last()[['year', 'dist_m']]
)
start_points = {
    loc_id: {
        'last_dist': row['dist_m'],
        'last_year': int(row['year']),
        'v_hist':    model_velocities.get(loc_id, 0.0),
    }
    for loc_id, row in last_obs.iterrows()
}
print(f'Start points: {len(start_points):,} locations')

# --- Predict ---
predicted_dist_df = predict_iterative_baseline(
    model_velocities, start_points,
    start_year=START_YEAR, end_year=END_YEAR, step=STEP,
)
print(f'Predicted dist: {len(predicted_dist_df):,} rows ({predicted_dist_df["location_id"].nunique():,} locations)')
print(predicted_dist_df.head(10))

## Step 4: Predict coordinates from dist

Use `offset_toward_points` (same as for historical dist): offset centerline by predicted_dist toward the bank side.

In [ ]:
ref_geom_lookup = build_ref_geom_lookup(bank_points, n_points=N_POINTS_FOR_DIST)

In [ ]:
predicted_points = []
for row in predicted_dist_df.itertuples(index=False):  # faster than iterrows
    loc_id = row.location_id
    dist = row.predicted_dist_m
    cline = cl_lookup.get(loc_id)
    ref_geom = ref_geom_lookup.get(loc_id)
    
    point = (
        point_from_offset(cline, dist, ref_geom)
        if (cline is not None and ref_geom is not None and len(ref_geom) > 0)
        else None
    )
    
    predicted_points.append({
        'location_id': loc_id,
        'year': row.year,
        'predicted_dist_m': dist,
        'velocity_m_per_yr': row.velocity_m_per_yr,
        'geometry': point,
    })


predicted_bank_positions = gpd.GeoDataFrame(predicted_points, geometry='geometry', crs=bank_points.crs)

plot_ids = [lid for lid in selected if lid in predicted_bank_positions['location_id'].values]
if not plot_ids:
    plot_ids = predicted_bank_positions['location_id'].unique()[:4].tolist()

### Step 4b: Plot scope regions with historical + predicted points (2026–2035)

In [ ]:
legend_extra_pred = [
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='gray',
    markeredgecolor='black', markersize=8, label='predicted'),
]

plot_regions(plot_ids, title='Scope regions: historical + predicted',
    legend_handles=year_patches + pred_patches + legend_extra_pred,
    draw_kwargs=dict(show_predictions=True)
)

### Step 4c: NVO regions with signaleringslijn

Plot a few NVO regions (from vvr_rates_of_change) with the signaleringslijn (Vlak_vrije_ruimte_natuurvriendelijke_oever_ln) overlaid.

In [ ]:
# Load signaleringslijn (NVO boundary)
SIGNALERING_GPKG = DATA_DIR / '01_raw/scope/20260205_signaleringslijn.gpkg'
SIGNALERING_LAYER = 'Vlak_vrije_ruimte_natuurvriendelijke_oever_ln'
signaleringslijn = gpd.read_file(SIGNALERING_GPKG, layer=SIGNALERING_LAYER)
if signaleringslijn.crs.to_epsg() != 28992:
    signaleringslijn = signaleringslijn.to_crs(28992)
VVR_COLOR = '#7b2d8b'  # purple (same as explore_point_selection)

nvo_location_ids = get_nvo_location_ids(vvr, scope_raw)
print(f'NVO locations (from vvr_rates_of_change): {len(nvo_location_ids):,}')

# Pick NVO regions that have predictions
nvo_with_pred = [lid for lid in nvo_location_ids if lid in predicted_bank_positions['location_id'].values]
nvo_plot_ids = pick_across_clusters(nvo_with_pred, n=4)
print(f'Plotting NVO regions: {nvo_plot_ids}')


In [ ]:
legend_handles = year_patches + pred_patches + [
    plt.Line2D([0], [0], color='black', lw=2.5, label='centreline'),
    plt.Line2D([0], [0], color=VVR_COLOR, lw=2.5, label='signaleringslijn'),
    mpatches.Patch(fc='#f0f0f0', ec='#aaaaaa', label='scope'),
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='gray',
               markeredgecolor='black', markersize=8, label='predicted'),
]

plot_regions(nvo_plot_ids, title='NVO regions: historical + predicted + signaleringslijn',
    legend_handles=legend_handles,
    draw_kwargs=dict(show_predictions=True, show_signalering=True)
)

### Step 4d: VVR shape consistency — centerline + signaleringslijn + parallel line

Plot ~50 scope regions with centerline, VVR (signaleringslijn), and a **parallel line** at `dist_signaleringslijn` distance.
`dist_signaleringslijn(centerline, vvr_geom)` returns the minimum perpendicular distance from CL to VVR; the parallel line is offset by that distance toward the VVR.
Goal: assess how consistent the VVR shape is and whether a parallel-line approximation can support a general "boundary exceeded" algorithm.

In [ ]:
from shapely import offset_curve
from shapely.geometry import Point, LineString
import math

nvo_with_cl = [lid for lid in nvo_location_ids if lid in cl_lookup]
vvr_consistency_ids = pick_across_clusters(nvo_with_cl, n=50)

PARALLEL_LINE_COLOR = '#e67e22'

legend_handles = [
    plt.Line2D([0], [0], color='black', lw=2, label='centreline'),
    plt.Line2D([0], [0], color=VVR_COLOR, lw=2, label='VVR (signaleringslijn)'),
    plt.Line2D([0], [0], color=PARALLEL_LINE_COLOR, lw=2, ls='--', label='parallel @ dist_signaleringslijn'),
    mpatches.Patch(fc='#f0f0f0', ec='#aaaaaa', label='scope'),
]

plot_regions(
    vvr_consistency_ids[:50],
    title='VVR shape consistency: centreline + signaleringslijn only (50 regions)',
    legend_handles=legend_handles,
    draw_kwargs=dict(show_bank_points=False, show_signalering=True, show_parallel_line=True),
    compact=True,
    n_cols=10,
)

---
## Step 5 & 6: Distance to signaleringslijn + Time to crossing

For each NVO region: compute distance from centerline to signaleringslijn (VVR boundary), then find when the predicted bank distance exceeds it. Uses linear interpolation between prediction years.

In [ ]:
# scope_raw has vlakken_scope geometry per location_id
vvr_crossing = compute_vvr_crossing_year(
    predicted_dist_df,
    nvo_location_ids,
    centerlines,
    scope_raw,
    signaleringslijn,
)
print(f'VVR crossing: {len(vvr_crossing)} NVO regions')
crossed = vvr_crossing[vvr_crossing['crossing_year'].notna()]
print(f'Crossing within prediction window: {len(crossed)} regions')

In [ ]:
display(vvr_crossing.head(20))

In [ ]:
vvr_crossing['years_to_crossing'].value_counts()


---
## Step 7: Export to GeoPackage

Creates a **full copy** of the base GeoPackage (never overwrites the original),
then appends two new layers:

| Layer | Content |
|---|---|
| `predicted_bank_positions` | GeoPoint — one row per (location_id, year), with predicted dist and velocity |
| `predicted_vvr_crossing` | Polygon (scope geometry) — NVO regions with crossing_year, years_to_crossing, dist_to_vvr_m |

**Note on `vvr_rates_of_change`:** that layer has no `location_id` and a 1:many spatial
relationship with scope polygons, so we write crossing results as a separate named layer
instead of patching the existing one.

In [ ]:
import shutil
from datetime import date

# --- Config ---
OUTPUT_NAME = f"wocu_erosion_predictions_{date.today().strftime('%Y%m%d')}.gpkg"
OUTPUT_GPKG = DATA_DIR / f"02_processed/erosion/{OUTPUT_NAME}"

# Copy the base GeoPackage (never modify the original)
shutil.copy2(POSTPROC_GPKG, OUTPUT_GPKG)
print(f"Copied base GeoPackage to: {OUTPUT_GPKG.name}")

In [ ]:
# --- Layer 1: predicted_bank_positions (GeoPoint) ---
# predicted_bank_positions already is a GeoDataFrame from Step 4
bank_pos_export = predicted_bank_positions[
    predicted_bank_positions.geometry.notna()
].copy()
bank_pos_export = bank_pos_export.to_crs(28992)

bank_pos_export.to_file(OUTPUT_GPKG, layer='predicted_bank_positions', driver='GPKG')
print(f"Written layer 'predicted_bank_positions': {len(bank_pos_export):,} rows")

In [ ]:
# --- Layer 2: predicted_vvr_crossing (Polygon — scope geometry for NVO regions) ---
# Join vvr_crossing results onto scope_raw polygons via location_id
crossing_cols = ['location_id', 'dist_to_vvr_m', 'crossing_year',
                 'velocity_m_per_yr', 'years_to_crossing']
crossing_data = vvr_crossing[crossing_cols].copy()

# scope_raw has vlakken_scope geometry per location_id (used in compute_vvr_crossing_year)
scope_raw_indexed = scope_raw.set_index('location_id')[['geometry']].copy()

crossing_gdf = scope_raw_indexed.join(
    crossing_data.set_index('location_id'), how='inner'
).reset_index()
crossing_gdf = gpd.GeoDataFrame(crossing_gdf, geometry='geometry', crs=scope_raw.crs)
crossing_gdf = crossing_gdf.to_crs(28992)

crossing_gdf.to_file(OUTPUT_GPKG, layer='predicted_vvr_crossing', driver='GPKG')
print(f"Written layer 'predicted_vvr_crossing': {len(crossing_gdf):,} NVO regions")
print(f"  of which with crossing_year: {crossing_gdf['crossing_year'].notna().sum():,}")
print(f"  crossing_year range: {int(crossing_gdf['crossing_year'].min())} – {int(crossing_gdf['crossing_year'].max())}")

In [ ]:
# --- Verify: list all layers in the output file ---
from pyogrio import list_layers
layers_out = list_layers(OUTPUT_GPKG)
print(f"\nOutput: {OUTPUT_GPKG.name}")
print("Layers:")
for name, geom in layers_out:
    gdf = gpd.read_file(OUTPUT_GPKG, layer=name)
    print(f"  {name:40s} [{geom or 'None':15s}]  {len(gdf):>6,} rows  cols: {list(gdf.columns)[:5]}")
print(f"\nFile size: {OUTPUT_GPKG.stat().st_size / 1e6:.1f} MB")